# Notebook 05 — Article Charts

**Purpose:** Generate publication-ready charts for the Kalshi exotics article from live Dune data.

**Outputs:** PNG files in `../charts/` folder, ready to embed in the article.

**Charts produced:**
1. Monthly handle bar chart
2. Cumulative handle + notional dual-line
3. Implied probability over time (avg + median weekly)
4. Bet size distribution (log-scale buckets)
5. Fee revenue by product

In [ ]:
import os
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

# Setup
API_KEY = "YOUR_DUNE_API_KEY"
headers = {"X-Dune-API-Key": API_KEY}

CHARTS_DIR = Path("../charts")
CHARTS_DIR.mkdir(exist_ok=True)

# Branding
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica Neue", "Arial", "DejaVu Sans"],
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

PRIMARY = "#1f4e3d"      # forest green
SECONDARY = "#c47e2c"    # warm amber
NEUTRAL = "#6b6b6b"

def fetch(query_id):
    r = requests.get(f"https://api.dune.com/api/v1/query/{query_id}/results", headers=headers)
    return pd.DataFrame(r.json()["result"]["rows"])

print("Setup complete")

## Chart 1 — Monthly Handle Bar Chart

The growth story. $82K in Sep 2025 to $389M in May 2026.

In [ ]:
df = fetch("7521948")
df["month"] = pd.to_datetime(df["month"])
df = df.sort_values("month")
df["handle_m"] = df["handle"] / 1e6

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(df["month"].dt.strftime("%b %Y"), df["handle_m"], color=PRIMARY, edgecolor="white", linewidth=1.5)

# Labels above bars
for bar, val in zip(bars, df["handle_m"]):
    if val >= 1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(df["handle_m"])*0.01,
                f"${val:,.0f}M", ha="center", va="bottom", fontsize=9, color="#333")

ax.set_title("Monthly Exotic Handle — Kalshi Combos", fontsize=15, fontweight="bold", pad=15, loc="left")
ax.set_ylabel("Handle (USD millions)", fontsize=11)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}M"))
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=30)
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.savefig(CHARTS_DIR / "01_monthly_handle.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"\nLatest month: {df.iloc[-1]['month'].strftime('%b %Y')} — ${df.iloc[-1]['handle_m']:,.1f}M")

## Chart 2 — Cumulative Handle + Notional Dual-Line

In [ ]:
df = fetch("7521948")
df["month"] = pd.to_datetime(df["month"])
df = df.sort_values("month")
df["cum_notional_b"] = df["cumulative_notional"] / 1e9
df["cum_handle_b"] = df["cumulative_handle"] / 1e9

fig, ax = plt.subplots(figsize=(11, 6))
ax.fill_between(df["month"], df["cum_notional_b"], color=PRIMARY, alpha=0.15, label="Cumulative Notional")
ax.plot(df["month"], df["cum_notional_b"], color=PRIMARY, linewidth=2.5, label="Notional")
ax.plot(df["month"], df["cum_handle_b"], color=SECONDARY, linewidth=2.5, label="Handle (cash)")
ax.fill_between(df["month"], df["cum_handle_b"], color=SECONDARY, alpha=0.2)

# Annotate endpoints
last = df.iloc[-1]
ax.annotate(f"${last['cum_notional_b']:.1f}B", xy=(last["month"], last["cum_notional_b"]),
            xytext=(8, 0), textcoords="offset points", fontsize=11, fontweight="bold", color=PRIMARY)
ax.annotate(f"${last['cum_handle_b']:.2f}B", xy=(last["month"], last["cum_handle_b"]),
            xytext=(8, 0), textcoords="offset points", fontsize=11, fontweight="bold", color=SECONDARY)

ax.set_title("Cumulative Handle vs Notional Since Launch", fontsize=15, fontweight="bold", pad=15, loc="left")
ax.set_ylabel("USD billions", fontsize=11)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:.1f}B"))
ax.legend(loc="upper left", frameon=False, fontsize=11)
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.savefig(CHARTS_DIR / "02_cumulative_handle_notional.png", dpi=200, bbox_inches="tight")
plt.show()

## Chart 3 — Implied Probability Drift (Weekly)

The 20% → 9% story.

In [ ]:
df = fetch("7516976")
df["week"] = pd.to_datetime(df["week"])
df = df.sort_values("week")

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(df["week"], df["avg_price_pct"], color=PRIMARY, linewidth=2.5, label="Average")
ax.plot(df["week"], df["median_price_pct"], color=SECONDARY, linewidth=2.5, linestyle="--", label="Median")

# Annotate start and end
first, last = df.iloc[0], df.iloc[-1]
ax.annotate(f"{first['avg_price_pct']:.1f}%", xy=(first["week"], first["avg_price_pct"]),
            xytext=(5, 8), textcoords="offset points", fontsize=11, fontweight="bold", color=PRIMARY)
ax.annotate(f"{last['avg_price_pct']:.1f}%", xy=(last["week"], last["avg_price_pct"]),
            xytext=(5, 8), textcoords="offset points", fontsize=11, fontweight="bold", color=PRIMARY)

ax.set_title("Implied Probability per Trade — Weekly", fontsize=15, fontweight="bold", pad=15, loc="left")
ax.set_ylabel("Implied probability (%)", fontsize=11)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.legend(loc="upper right", frameon=False, fontsize=11)
ax.grid(axis="x", visible=False)
ax.set_ylim(0, max(df["avg_price_pct"].max(), df["median_price_pct"].max()) * 1.2)

plt.tight_layout()
plt.savefig(CHARTS_DIR / "03_implied_probability_drift.png", dpi=200, bbox_inches="tight")
plt.show()

## Chart 4 — Bet Size Distribution (Log-Scale Buckets)

Most bets are under $10. A thin tail does the work.

In [ ]:
sql = """
SELECT
    CASE
        WHEN contracts_traded * price / 100.0 < 2    THEN '1: Under $2'
        WHEN contracts_traded * price / 100.0 < 5    THEN '2: $2-$5'
        WHEN contracts_traded * price / 100.0 < 10   THEN '3: $5-$10'
        WHEN contracts_traded * price / 100.0 < 20   THEN '4: $10-$20'
        WHEN contracts_traded * price / 100.0 < 50   THEN '5: $20-$50'
        WHEN contracts_traded * price / 100.0 < 200  THEN '6: $50-$200'
        ELSE                                              '7: $200+'
    END AS bucket,
    COUNT(*) AS num_trades,
    ROUND(SUM(contracts_traded * price / 100.0), 0) AS total_cash
FROM kalshi.trade_report
WHERE starts_with(report_ticker, 'KXMVE')
  AND contracts_traded > 0 AND price > 0
GROUP BY 1
ORDER BY 1
"""

# Use existing query 7550808 results would only give us percentiles
# Run the bucket query fresh
import time
r = requests.post('https://api.dune.com/api/v1/query', headers={**headers, 'Content-Type': 'application/json'},
                  json={'name': 'NB05 Bet Size Buckets', 'query_sql': sql})
qid = r.json()['query_id']
r = requests.post(f'https://api.dune.com/api/v1/query/{qid}/execute', headers=headers)
eid = r.json()['execution_id']
while True:
    time.sleep(4)
    if requests.get(f'https://api.dune.com/api/v1/execution/{eid}/status', headers=headers).json()['state'] == 'QUERY_STATE_COMPLETED':
        break

rows = requests.get(f'https://api.dune.com/api/v1/execution/{eid}/results', headers=headers).json()['result']['rows']
df = pd.DataFrame(rows).sort_values("bucket")
df["label"] = df["bucket"].str.split(": ", expand=True)[1]
df["trades_m"] = df["num_trades"] / 1e6
df["pct_trades"] = df["num_trades"] / df["num_trades"].sum() * 100
df["pct_cash"] = df["total_cash"] / df["total_cash"].sum() * 100

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(df["label"], df["pct_trades"], color=PRIMARY, edgecolor="white", linewidth=1.5)

for bar, pct in zip(bars, df["pct_trades"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{pct:.1f}%", ha="center", va="bottom", fontsize=10, color="#333")

ax.set_title("Bet Size Distribution — % of Trades", fontsize=15, fontweight="bold", pad=15, loc="left")
ax.set_ylabel("% of all trades", fontsize=11)
ax.set_xlabel("Bet size (USD)", fontsize=11)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.savefig(CHARTS_DIR / "04_bet_size_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

## Chart 5 — Fee Revenue by Product

Two products = 95% of estimated Kalshi fees.

In [ ]:
df = fetch("7576002").sort_values("Est. Kalshi Fee Revenue ($)", ascending=True)
df["fee_m"] = df["Est. Kalshi Fee Revenue ($)"] / 1e6

# Group small ones as "Other"
top = df[df["fee_m"] >= 1].copy()
other_total = df[df["fee_m"] < 1]["fee_m"].sum()
if other_total > 0:
    top = pd.concat([
        pd.DataFrame([{"Product": "Other (8 products)", "fee_m": other_total}]),
        top
    ], ignore_index=True)

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(top["Product"], top["fee_m"], color=PRIMARY, edgecolor="white", linewidth=1.5)

for bar, val in zip(bars, top["fee_m"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f"${val:,.1f}M", va="center", fontsize=10, color="#333", fontweight="bold")

ax.set_title("Est. Kalshi Fee Revenue by Product (Since Launch)", fontsize=15, fontweight="bold", pad=15, loc="left")
ax.set_xlabel("Estimated fees (USD millions)", fontsize=11)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:.0f}M"))
ax.grid(axis="y", visible=False)

# Add total annotation
total_fees = df["fee_m"].sum()
ax.text(0.98, 0.05, f"Total: ${total_fees:,.1f}M", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=12, fontweight="bold", color=PRIMARY,
        bbox=dict(boxstyle="round,pad=0.5", facecolor="white", edgecolor=PRIMARY, alpha=0.9))

plt.tight_layout()
plt.savefig(CHARTS_DIR / "05_fee_revenue_by_product.png", dpi=200, bbox_inches="tight")
plt.show()